<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 6–7 扩展实验：Schema 变化、导入删除与冲突检测</h1>
<p>目标 Doris 4.1.3 · 独立 ext_* 实验表</p>
</div>

[扩展入口](README.md) · [主线学习目录](../README.md)

先完成主线 Lab 6、7。仅重建 `ext_schema`、`ext_delete_sign`、`ext_event_stage`；不改 orders_clean/current 或业务历史。
建议 25–35 分钟。只在课程沙箱单内核执行，保留结果供排查。
验收：加列后旧值明确、下游显式列投影不变；删除后旧版本不能复活订单；相同事件 ID 的不同内容被检测。
这里没有实现真实 Binlog 恢复，也没有提供多消费者并发下原子性的冲突拒收服务。

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. 加列、兼容性与异步变更状态

把十笔订单复制到独立表，先记录下游显式列投影。新增可空渠道列，旧记录为 NULL。
再将 amount_cents 从 INT 改为 BIGINT，等待 SHOW ALTER TABLE COLUMN 的任务完成后核对结果。
轻量/重型取决于操作和目标版本；不能用本次耗时推导所有 Schema Change 都只改元数据。

In [ ]:
import time
lab.execute("DROP TABLE IF EXISTS ext_schema")
lab.execute('''CREATE TABLE ext_schema (order_id BIGINT, amount_cents INT)
DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_schema SELECT order_id,CAST(order_amount*100 AS INT) FROM orders_clean")
expected = lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id")
expect(lab.query("SELECT COUNT(*),SUM(amount_cents) FROM ext_schema"), [(10,140000)])
for statement, column, target_type in (("ALTER TABLE ext_schema ADD COLUMN order_channel VARCHAR(20) NULL", "order_channel", "varchar"),
                                      ("ALTER TABLE ext_schema MODIFY COLUMN amount_cents BIGINT", "amount_cents", "bigint")):
    show_sql("Schema Change",statement)
    lab.execute(statement)
    deadline = time.monotonic()+120
    while True:
        with lab.connection.cursor() as cursor:
            cursor.execute("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema' ORDER BY CreateTime DESC LIMIT 1")
            fields = [c[0] for c in cursor.description]
            jobs = [dict(zip(fields,row)) for row in cursor.fetchall()]
        types = {row[0]: row[1].lower() for row in lab.query("DESC ext_schema")}
        if types.get(column, "").startswith(target_type) and (not jobs or jobs[0]["State"] == "FINISHED"):
            break
        if jobs and jobs[0]["State"] == "CANCELLED":
            raise RuntimeError(str(jobs))
        if time.monotonic()>=deadline:
            raise TimeoutError(str(jobs))
        time.sleep(1)
    lab.sql("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema'")
    expect(lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id"), expected)
expect(lab.query("SELECT COUNT(*),COUNT(order_channel) FROM ext_schema"), [(10,0)])
lab.sql("DESC ext_schema")

## 2. 导入删除标记和旧版本重放

使用 Unique Key + Sequence 的独立三列表。请求用 MERGE 和 DELETE ON 条件把操作类型映射成删除标记。
先导入 CREATED v1，随后删除 v2，再重放 v1，最后明确写入新版本 PAID v3。
验收顺序为 1 → 0 → 0 → 1 行；新版本重建是显式新事件，不是旧记录随机复活。
普通查询不可见不等于立即释放磁盘；此处不改业务退款表。

In [ ]:
import os
import requests
from uuid import uuid4
lab.execute("DROP TABLE IF EXISTS ext_delete_sign")
lab.execute('''CREATE TABLE ext_delete_sign (
order_id BIGINT NOT NULL,status VARCHAR(20) NOT NULL,event_version BIGINT NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1","enable_unique_key_merge_on_write"="true","function_column.sequence_col"="event_version")''')
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
for payload, expected in (("900001,CREATED,1,UPSERT\n",[(900001,"CREATED",1)]),
                          ("900001,CREATED,2,DELETE\n",[]),
                          ("900001,CREATED,1,UPSERT\n",[]),
                          ("900001,PAID,3,UPSERT\n",[(900001,"PAID",3)])):
    headers = {"label":"ext_delete_"+uuid4().hex,"format":"csv","column_separator":",",
               "columns":"order_id,status,event_version,op","merge_type":"MERGE","delete":"op='DELETE'",
               "group_commit":"off_mode","strict_mode":"true","max_filter_ratio":"0"}
    response = requests.put(f"{endpoint}/api/{lab.database}/ext_delete_sign/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload.encode(),
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    print(result)
    expect(result["Status"],"Success")
    expect(lab.query("SELECT order_id,status,event_version FROM ext_delete_sign ORDER BY order_id"),expected)

## 3. 相同事件 ID，内容却变了

Unique Key(event_id) 可以覆盖同 ID 的记录，但不自动判断内容冲突。先把原始候选事件放入独立暂存表。
重复内容仍符合幂等契约；同 ID 不同内容必须停下来，不能直接写入主线历史或当前表。
本例只保留 order_id/status/version 三个业务字段，生产规则应覆盖完整规范化业务载荷。
单写者先检查再写的示范不提供多写者事务保障。

In [ ]:
from dw_course.runtime import expected_failure
lab.execute("DROP TABLE IF EXISTS ext_event_stage")
lab.execute('''CREATE TABLE ext_event_stage (
event_id VARCHAR(32),order_id BIGINT,status VARCHAR(20),event_version BIGINT
) DUPLICATE KEY(event_id) DISTRIBUTED BY HASH(event_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'PAID',2),('CONFLICT_DEMO',900001,'PAID',2)")
conflicts = """SELECT event_id FROM (
SELECT DISTINCT event_id,order_id,status,event_version FROM ext_event_stage
) v GROUP BY event_id HAVING COUNT(*)>1 ORDER BY event_id"""
expect(lab.query(conflicts),[])
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'CANCELLED',2)")
with expected_failure("事件内容冲突", "同 ID 的不同内容已识别，停止后续业务写入"):
    expect(lab.query(conflicts),[])
expect(lab.query(conflicts),[("CONFLICT_DEMO",)])
lab.sql("SELECT * FROM ext_event_stage ORDER BY event_id,status")

## 4. 独立解释与排查

解释渠道列为何可以是 NULL，哪些下游 SELECT * 或按位置写入会受新列影响。
解释删除版本 2 为什么能挡住旧版本 1，为什么明确的新版本 3 又能出现。
尝试只改变暂存记录的订单号，说明冲突检测为何仍能发现错误。
Schema 变更失败查看 SHOW ALTER TABLE COLUMN；导入失败检查响应和 ErrorURL；冲突保留全部候选记录，不能用覆盖消除证据。

参考：[Schema Change](https://doris.apache.org/docs/4.x/table-design/schema-change/)、
[更新与删除](https://doris.apache.org/docs/4.x/data-operate/update/update-overview/)。

In [ ]:
lab.close()